# カスタムLoRA: LLaMA3-8B-Instructのファインチューニング
---

**概要:** 

このNotebookでは、Low-Rank Adaptation (LoRA)を使ってLLaMA 3モデルをファインチューニングするプロセスを探ります。扱うトピックは以下の通りです：

1. **データ前処理**: モデルのトレーニングに最適な入力を確保するためのデータの準備とクリーニング
2. **学習のセットアップ**:
   - **Tokenizerのロード**: テキストデータを処理するために、LLaMA 3に適したトークナイザーを初期化
   - **ハイパーパラメータの設定**: 効果的なモデルトレーニングのための主要なハイパーパラメータを定義
   - **PEFT (Parameter-Efficient Fine-Tuning)**:  少ない計算ソースでモデルをファインチューニングする技術
   - **量子化**: パフォーマンスを維持したままモデルサイズを縮小し、より高速な推論とメモリ使用量の削減が可能
   - **事前学習済みモデルのロード**: 事前学習されたLLaMA 3モデルを持ち込み、カスタムデータでさらにファインチューニング
   - **Trainerのハイパーパラメータの設定**: 最適なパフォーマンスを得るために、トレーニングループに特有のパラメータを設定
   - **推論の実行**: ファインチューニングされたモデルを新しいデータで評価し、その性能と精度をテストを実施

このNotebookは、LlaMA 3モデルを特定のユースケースに合わせて効果的にファインチューニングし、最適化するための手順を説明します。

## データ前処理
トレーニングにはLLM-jpの[magpie-sft-v1.0](https://huggingface.co/datasets/llm-jp/magpie-sft-v1.0)データセット(Apache 2.0ライセンス)を使用します。

In [1]:
import warnings
warnings.filterwarnings("ignore")
from datasets import load_dataset

In [ ]:
# 日本語の指示データセット
dataset_name = "llm-jp/magpie-sft-v1.0"
ds = load_dataset(dataset_name) # ダウンロードには15秒程度かかります 
ds

データセットから5つのサンプルを見てみましょう。`conversations`フィールドに`user`と`assistant`の会話データが含まれています。

In [ ]:
for sample in ds['train'].select(range(5)):
    print(f"\n {'*' * 64}\n{sample}\n{'*' * 64}")

Llama3モデルは，学習データセットが[こちら](https://llama.meta.com/docs/model-cards-and-prompt-formats/meta-llama-3/)のような特定の形式である必要があります（＝`<|begin_of_text|>`や`<|eot_id|>`などの特殊トークンを使用）。これは、トークナイザの`apply_chat_template`メソッドを使用して、データセットの`conversations`フィールドを変換することで実現できます。サンプルを１つ見てみましょう。

Llama-3ファミリーのモデルはオープンソースですが、アクセスリクエストの承認が必要です。ブートキャンプ環境では、ウエイトはすでにhuggingface互換フォーマットに変換され、参加者が素早くアクセスできるように共有の場所に保存されています。

ご自身の環境でこの教材を実行する場合は、[こちら](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)からLlamaモデルへのアクセスをリクエストし、こちらの[リンク](https://huggingface.co/settings/tokens)からHuggingFaceユーザアクセストークンを生成してください。

In [ ]:
from transformers import AutoTokenizer

# 独自の環境で実行する場合は、以下で`token`を指定してください。
# token='hf_..'
# base_model = "meta-llama/Meta-Llama-3-8B-Instruct"

# 共有モデルの重みの場所
base_model = "/mnt/lustre/docker-strg/llama3_8b_weights" 

tokenizer = AutoTokenizer.from_pretrained(base_model) # Tokenizerをロード

# conversationsフィールドをLlama3のフォーマットに変換
preprocessed_text = tokenizer.apply_chat_template(
    conversation=ds['train'][0]['conversations'],
    tokenize=False,
    # token=token, # 独自の環境で実行する場合は、こちらを指定してください。
)
print(preprocessed_text)

データセット内の全てのサンプルに対して同じ操作を行います。これはデータセットの`map`メソッドを使用して実現できます。変換されたテキストデータを`preprocessed_conversations`フィールドに保存します。

In [5]:
ds['train'] = ds['train'].map(lambda x: {'preprocessed_conversations': tokenizer.apply_chat_template(
    conversation=x['conversations'],
    tokenize=False, 
)})

`preprocessed_conversations`フィールドの最初のサンプルを見てみましょう。

In [ ]:
print(ds['train']['preprocessed_conversations'][0])

このデータセットを保存する前に、データセットをトレーニングセットとテストセットに分割します。

In [ ]:
ds = ds['train'].train_test_split(test_size=0.1, seed=42)
ds

In [ ]:
!mkdir -p data/ds_preprocess
ds.save_to_disk('data/ds_preprocess/')

これで変換されたデータセットを保存できました。これで、カーネルが故障した場合にデータセットを直接ロードするのに役立ちます。

## 学習セットアップ

ファインチューニングの設定には、以下のライブラリーが役に立ちます。

In [9]:
# メモリが不足している場合は、`os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"`のコメントを解除してください。

import torch

from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from datasets import load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

# import os
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

重要な成果物のロードと保存のための重要なパスを設定します。

In [10]:
# LoRAのベースモデル
# base_model = "meta-llama/Meta-Llama-3-8B-Instruct" # 独自の環境で実行する場合はこちらを使用してください。
base_model = "/mnt/lustre/docker-strg/llama3_8b_weights" # 共有モデルの重みの場所

# 学習データセットのパス
data_path = "data/ds_preprocess/train"
# テストデータセットのパス
eval_path = "data/ds_preprocess/test"

# 変換されたデータセットのロード
dataset = load_from_disk(data_path)
eval_dataset = load_from_disk(eval_path)

先ほど保存したデータセットが正確にロードされていることを確認します。（上記で確認したサンプルとは異なるサンプルが表示されている可能性がありますが、問題ありません。）

In [ ]:
print(dataset['preprocessed_conversations'][0])

### Tokenizerのロード

In [12]:
# Llama-3-8B-Instructのトークナイザーをダウンロードします。
tokenizer = AutoTokenizer.from_pretrained(base_model,
                                          # token=token,
                                          # trust_remote_code=True
                                         )

In [13]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

結果を保存するディレクトリを作成します。

In [14]:
! mkdir -p model

### 学習のハイパーパラメータの設定

In [15]:
# ハイパーパラメータの設定
training_params = TrainingArguments(
    output_dir="model/results",             # モデルの結果を保存するディレクトリ
    num_train_epochs=2,                     # トレーニングエポック数
    per_device_train_batch_size=5,          # 学習中のデバイスあたりのバッチサイズ
    gradient_accumulation_steps=4,          # 勾配を蓄積するステップ数
    group_by_length=True,                   # 同様の長さの文をグループ化
    save_steps=100,                         # 100ステップごとにモデルのチェックポイントを保存
    logging_steps=25,                       # 25ステップごとにメトリクスをログに記録
    learning_rate=2e-4,                     # 初期学習率
    weight_decay=0.001,                     # 適用する重み減衰（L2正則化）
    evaluation_strategy="steps",            # 一定のステップ数で評価を実施
    eval_steps=25,                          # 25ステップごとに評価
    fp16=False,                             # FP16
    bf16=False,                             # BF16
    max_grad_norm=0.3,                      # 最大勾配ノルム（勾配クリッピング用）
    max_steps=-1,                           # 学習ステップの総数（-1は制限なしを意味します）
    warmup_ratio=0.03,                      # 学習率のウォームアップを実行するステップの割合
    optim="paged_adamw_32bit",              # 使用するオプティマイザ（ページングメモリ付き32ビットAdamW）
    lr_scheduler_type="constant",           # 学習率スケジューラタイプ（定数）
    report_to="tensorboard"                 # レポートツール（この場合はTensorBoard）
)


### PEFT
LoRA テクニックは `LoraConfig` によって適用され、ベースモデルへの適用方法を制御する PEFT パラメータを提供します。以下のセルで使用されているパラメータの説明は以下の通りです：

- **lora_alpha**： LoRA スケーリング係数
- **lora_dropout**： LoRAレイヤーのドロップアウト確率。
- **r**: 更新行列のランク、int で表される。ランクが低いほど更新行列が小さくなり、学習可能なパラメータが少なくなります。
- **bias**： バイアスパラメータを学習するかどうかを指定する。'none'、'all'、'lora_only'のいずれかを指定します。
- **task_type**： `CAUSAL_LM`, `FEATURE_EXTRACTION`, `QUESTION_ANS`, `SEQ_2_SEQ_LM`, `SEQ_CLS and TOKEN_CLS` などがあります。
- **target_modules**: LoRAの適用対象のモジュールを指定します。`all-linear`はすべての線形層にLoRAを適用します。  

実行したいタスクはテキスト生成なので、task_type をテキスト生成タスクでよく使われる Causal language model `(CAUSAL_LM)` に設定しました。以下のセルを実行して、LoRAの設定を行ってください。

In [16]:
peft_params = LoraConfig(
    lora_alpha=16,                # LoRAスケーリング係数
    lora_dropout=0.1,             # LoRAレイヤーのドロップアウト確率
    r=32,                         # LoRA行列のランク
    bias="none",                  # 適用するバイアスの種類（この場合は "none"）
    task_type="CAUSAL_LM",        # タスクの種類（この場合はCausal Language Modeling）
    target_modules="all-linear",  # LoRAを適用するモジュール（全線形層）
)


### 量子化
**4-bit 量子化コンフィグレーション**

モデルの量子化は、一般的なディープラーニングの最適化手法であり、モデルデータ（ネットワークパラメータと活性化）を浮動小数点から低精度表現（通常は8ビット整数）に変換します。量子化はより少ないビット数でデータを表現するため、特に大規模な言語モデル（LLM）において、メモリ使用量を削減し、推論を高速化するのに有効な手法です。PEFTメソッドと組み合わせることで、推論のためのLLMの学習とロードを容易にすることができます。

<center><img src="imgs/quantization.png" height="400px" width="700px" /></center>
<center> <a href="https://developer.nvidia.com/blog/achieving-fp32-accuracy-for-int8-inference-using-quantization-aware-training-with-tensorrt/" > source: Using Quantization Aware Training with NVIDIA TensorRT</a></center>

モデルを量子化するいくつかの方法とアルゴリズムは[こちら](https://huggingface.co/docs/peft/main/en/developer_guides/quantization)にあります。量子化を簡単に実装し、Transformersと統合するためのライブラリに `bitsandbytes` ライブラリがあります。このライブラリは、`BitsAndBytesConfig`クラスを使ってモデルを8ビットまたは4ビットに量子化するための設定パラメータを提供します。以下のセルで使用されている4ビットのパラメータは以下のように記述されています：

- **load_in_4bit**: モデルをロードする際に4ビットに量子化するには `True` を設定。
- **bnb_4bit_quant_type**: `"nf4"` に設定すると、正規分布から初期化された重みに特別な4ビットのデータ型を使用。
- **bnb_4bit_use_double_quant**: `True` に設定すると、既に量子化された重みをネストした量子化スキームで量子化。
- **bnb_4bit_compute_dtype**: `torch.float16` または `torch.bfloat16` に設定。

以下のセルを実行して、モデルの4ビット量子化を設定。


In [17]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

### 事前学習モデルのロード

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map={"": 0},
    # token=token
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
!nvidia-smi

モデルがGPUにロードされていることが分かります。

### Trainerのハイパーパラメータのセット

モデルトレーナを開始するために、 [Supervised fine-tuning (SFT)](https://huggingface.co/docs/trl/en/sft_trainer)からトレーナオブジェクトを作成します。SFTは強化学習を使って変換言語モデルを学習するための統合変換ツール [Reinforcement Learning (TRL)](https://huggingface.co/docs/trl/en/index)の一部です。その他に、[Reward Modeling step (RM)](https://huggingface.co/docs/trl/en/reward_trainer) や [Proximal Policy Optimization (PPO)](https://arxiv.org/abs/1707.06347)があります。SFTトレーナーオブジェクトでは、モデル、学習データセット、PEFT設定オブジェクト、モデル・トークナイザー、学習引数パラメータを設定します。また、データセット内で使用するフィールド（`text`）も指定します。
**注意:** *単一のDGX A100 GPUで実行する場合は、`max_seq_length`の値を1024に変更するか、デフォルトのnoneに設定します。*

In [ ]:
len(dataset)

In [ ]:
eval_dataset

In [ ]:
# 時間の節約のため、1000のランダムサンプルを使用して学習します。評価は100サンプルで行います。
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset.shuffle(seed=42).select(range(1000)),
    eval_dataset = eval_dataset.select(range(100)),
    dataset_text_field="preprocessed_conversations",
    peft_config=peft_params,
    args=training_params,
    max_seq_length=1024,
    packing=False,
)

以下のセルを実行して、以下のSFT trainerオブジェクトの学習を行います。

In [ ]:
import jinja2
print(jinja2.__version__)


In [ ]:
trainer.train()

In [ ]:
# LoRAアダプターのsafetensorsファイルおよびトークナイザーを保存
new_model_name = "model/Llama-3-8b-instruct-hf-finetune"

trainer.model.save_pretrained(new_model_name)
trainer.tokenizer.save_pretrained(new_model_name)

In [26]:
# LoRAアダプターのバイナリファイルを保存
trainer.model.save_pretrained(new_model_name,safe_serialization=False)

### 推論の実行

まずは**学習前のベースのモデル**の生成結果を確認します。  

Llama-3-8B-Instructの例だと、日本語で入力しているのにもかかわらず、生成結果が英語になっています。
**CUDA Out of Memoryが出る場合は、一度Jupyterカーネルを再起動して以下のセルから実行してください。**

In [ ]:
from transformers import pipeline, AutoTokenizer

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

pp_base = pipeline(
    model=model_name, 
    tokenizer=tokenizer, 
    max_new_tokens=512, 
    task="text-generation", 
    temperature=.7, 
    top_p=0.4,
    device="cuda:0"
)
response = pp_base(tokenizer.apply_chat_template(
    conversation=[
        {"role": "user", "content": "最近朝起きるのが辛いのですが、その場合はどうすればいいですか？"}
    ],
    tokenize=False,
))[0]['generated_text']
print(response)

**想定される出力**
```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

最近朝起きるのが辛いのですが、その場合はどうすればいいですか？<|eot_id|>assistant

The struggle is real! 😴 Waking up early can be tough, but don't worry, I've got some tips to help you overcome the morning blues:

1. **Set a consistent wake-up time**: Set your alarm for the same time every day, including weekends. Consistency will help regulate your body's internal clock.
2. **Gradually adjust your wake-up time**: If you're not a morning person, don't try to wake up at 5 am immediately. Start by setting your alarm 15-30 minutes earlier each day and gradually work your way up to your desired wake-up time.
3. **Create a bedtime routine**: Wind down before bed with a relaxing activity, such as reading, meditation, or a warm bath. This will help signal to your body that it's time to sleep.
4. **Get enough sleep**: Aim for 7-9 hours of sleep each night to ensure you're well-rested and energized in the morning.
5. **Get some morning sunlight**: Exposure to natural light in the morning helps regulate your circadian rhythms, making it easier to wake up. Open your curtains or take a short walk outside in the morning.
6. **Avoid screens before bed**: The blue light emitted by smartphones, tablets, and computers can suppress melatonin production, making it harder to fall asleep and wake up in the morning.
7. **Use a wake-up light**: A wake-up light is a device that simulates a sunrise by gradually increasing the light in your room, helping to wake you up naturally.
8. **Make a plan for the morning**: Give yourself a reason to get out of bed by setting a goal or scheduling something enjoyable for the morning, such as exercise, breakfast with a friend, or a hobby.
9. **Use a motivational alarm**: Choose an alarm sound that motivates you to get out of bed, such as a favorite song or a motivational quote.
10. **Reward yourself**: Give yourself a small reward for waking up on time, such as a favorite breakfast treat or a short break from work.

Remember, it may take some time for your body to adjust to a new wake-up time. Be patient, and don't be too hard on yourself if you don't see immediate results. Good luck! 💪
```

続いて先ほど学習したLoRAモデルの生成結果を確認します。まずは、学習したモデルのLoRAアダプターをロードします。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

peft_model_path = "model/Llama-3-8b-instruct-hf-finetune"
peft_config = PeftConfig.from_pretrained(peft_model_path)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    quantization_config=quant_config,
    device_map={"": 0},
)
peft_model = PeftModel.from_pretrained(
    model, # modified inplace
    peft_model_path,
)

peft_model

学習したLoRAモデルの生成結果を確認します。まずはパイプラインを定義します。セルを実行した際に以下のWarningが出ますが、無視してください。

`The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalLM', 'MusicgenMelodyForCausalLM', 'MvpForCausalLM', 'NemotronForCausalLM', 'OlmoForCausalLM', 'OpenLlamaForCausalLM', 'OpenAIGPTLMHeadModel', 'OPTForCausalLM', 'PegasusForCausalLM', 'PersimmonForCausalLM', 'PhiForCausalLM', 'Phi3ForCausalLM', 'PLBartForCausalLM', 'ProphetNetForCausalLM', 'QDQBertLMHeadModel', 'Qwen2ForCausalLM', 'Qwen2MoeForCausalLM', 'RecurrentGemmaForCausalLM', 'ReformerModelWithLMHead', 'RemBertForCausalLM', 'RobertaForCausalLM', 'RobertaPreLayerNormForCausalLM', 'RoCBertForCausalLM', 'RoFormerForCausalLM', 'RwkvForCausalLM', 'Speech2Text2ForCausalLM', 'StableLmForCausalLM', 'Starcoder2ForCausalLM', 'TransfoXLLMHeadModel', 'TrOCRForCausalLM', 'WhisperForCausalLM', 'XGLMForCausalLM', 'XLMWithLMHeadModel', 'XLMProphetNetForCausalLM', 'XLMRobertaForCausalLM', 'XLMRobertaXLForCausalLM', 'XLNetLMHeadModel', 'XmodForCausalLM'].`

In [ ]:
from transformers import pipeline, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

# パイプラインの定義
pp_peft = pipeline(
    model=peft_model, 
    tokenizer=tokenizer,
    max_new_tokens=512, 
    task="text-generation", 
    temperature=.7, 
    top_p=0.4,
)

続いて生成結果です。

In [ ]:
response = pp_peft(tokenizer.apply_chat_template(
    conversation=[
        {"role": "user", "content": "最近朝起きるのが辛いのですが、その場合はどうすればいいですか？"}
    ],
    tokenize=False,
))[0]['generated_text']
print(response)

**想定される出力**
```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

最近朝起きるのが辛いのですが、その場合はどうすればいいですか？<|eot_id|>assistant

朝起きるのが辛いのは、体が自然と眠りにつく時間帯に起きる必要があるため、体調が整っていない状態で起きることによるものです。以下にいくつかの対策を提案します：

1. **リラックスした睡眠**: 7時間以上の睡眠を取ることで、朝起きるのが楽になることがあります。ただし、過度に過眠は避けてください。

2. **定時的な起床**: 一定の時間に起きるよう、闇時間を設定することで、体に慣れさせることができます。

3. **朝のリズムを作る**: 朝起きてすぐに何かを始めることは難しいかもしれませんが、まずはゆっくりとした動きを始めると良いでしょう。例えば、ゆっくりと体を動かす、または一日の計画を立てるなど、ゆっくりとした活動を始めることができます。

4. **朝食**: 朝食は体を温め、エネルギーを補給するのに役立ちます。健康的な朝食を摂ることで、朝起きるのが楽になるかもしれません。

5. **モチベーションを高める**: 朝起きるのが辛いと感じる場合、目標を設定することでモチベーションを高めることができます。例えば、特定のタスクを達成したい、または特定の時間に到着したいなど、具体的な目標を設定することで、朝起きるのが楽になるかもしれません。

これらの方法を試してみてください。何か他に質問があれば教えてください。何かお手伝いできることがあれば、喜んでお手伝いします。何か他に質問があれば教えてください。何かお手伝いできることがあれば、喜んでお手伝いします。何か他に質問があれば教えてください。何かお手伝いできることがあれば、喜んでお手伝いします。何か他に質問があれば教えてください。何かお手伝いできることがあれば、喜んでお手伝いします。何か他に質問があれば教えてください。何かお手伝いできることがあ
```

生成文が日本語になっており、LoRAチューニングによってモデルを日本語対応できたことが確認できます。さらに、こちらのモデルは重み部分は４ビットに量子化されているため、メモリの使用量がベースのモデルよりもおよそ1/4少なくなっている点も注目です。  

他にもいくつか生成してみましょう。

In [ ]:
response = pp_peft(tokenizer.apply_chat_template(
    conversation=[
        {"role": "user", "content": "NLPの分野ではどのような研究が行われていますか？"}
    ],
    tokenize=False,
))[0]['generated_text']
print(response)

In [ ]:
response = pp_peft(tokenizer.apply_chat_template(
    conversation=[
        {"role": "user", "content": "かわいいだけじゃだめですか？"}
    ],
    tokenize=False,
))[0]['generated_text']
print(response)

In [ ]:
# モデルとLoRAアダプターをマージしたい場合は以下のセルを実行してください。
model = PeftModel.from_pretrained(model, peft_model_path)
merged_model = model.merge_and_unload()
merged_model

In [ ]:
!nvidia-smi

以下のコマンドは、現在のカーネルを終了させ、NIMを実行するためにGPUを解放します。

In [ ]:
!kill -9 $(nvidia-smi --query-compute-apps=pid --format=csv,noheader | awk 'NR==1')


カスタムLoRAアダプターを使用するには、<a href="nim_lora_adapter.ipynb"> nim_lora_adapter</a> ノートブックをご参照ください。

---
## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.